# Land Cover Classification in the Meta River Basin — v6
### Sentinel-2 + Sentinel-1 + MapBiomas Colombia C3 · RF, 3D-2D CNN, ViT, Hybrid and Ensembles

## 0 · Configuration

In [ ]:
import os, sys, json, glob, time, datetime, random
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
for var, sub_dir in [('PROJ_DATA', 'share/proj'), ('PROJ_LIB', 'share/proj'), ('GDAL_DATA', 'share/gdal')]:
    if not os.environ.get(var) and os.path.isdir(os.path.join(sys.prefix, sub_dir)):
        os.environ[var] = os.path.join(sys.prefix, sub_dir)

PROFILE = 'colab_train' if IN_COLAB else 'pc_final'
PROFILES = {
    'pc_final':    dict(basin=1, gee=1, gee_stats=1, samples=0, windows=0, borders=0, flags=0, load=1, diagnostics=1, train=1, evaluate=1, maps=1, areas=1, final_map=1),
    'pc_full':     dict(basin=1, gee=1, gee_stats=1, samples=1, windows=1, borders=1, flags=1, load=1, diagnostics=0, train=1, evaluate=1, maps=1, areas=1, final_map=0),
    'pc_download': dict(basin=1, gee=1, gee_stats=1, samples=1, windows=1, borders=1, flags=1, load=1, diagnostics=0, train=0, evaluate=0, maps=0, areas=0, final_map=0),
    'colab_train': dict(basin=1, gee=0, gee_stats=0, samples=0, windows=0, borders=0, flags=0, load=1, diagnostics=0, train=1, evaluate=1, maps=0, areas=0, final_map=0),
    'pc_maps':     dict(basin=1, gee=1, gee_stats=0, samples=0, windows=1, borders=0, flags=0, load=1, diagnostics=0, train=0, evaluate=1, maps=1, areas=1, final_map=0),
}
RUN = {k: bool(v) for k, v in PROFILES[PROFILE].items()}
FORCE = set()

CFG = dict(
    ee_project       = 'hydro-496000',
    gpkg             = 'Meta.gpkg',
    train_years      = [2021, 2022, 2023],
    test_year        = 2024,
    pred_year        = 2025,
    n_per_class      = 1000,
    patch_radius     = 7,
    crs              = 'EPSG:32619',
    clear_threshold  = 0.60,
    max_cloud_scene  = 80,
    stats_scale      = 100,
    workers          = 32,
    chunk            = 500,
    tile             = 512,
    block_deg        = 0.2,
    epochs           = 80, patience = 10, batch = 256, lr = 1e-3,
    pred_batch       = 256,
    map_stride       = 3,
    seed             = 42,
    conf_min         = 0.70,
    mmu_ha           = 0.5,
    noise_ratio      = 10,
    target_precision = 0.85,
    ens_members      = 'auto',
    model_src        = '',
    border_frac      = 0.35,
    prior_corr       = 'mb_prev',
    em_floor         = 0.005,
    em_wmax          = 5.0,
    prior_alpha      = 0.5,
    area_boot        = 500,
    final_model      = 'auto',
    final_stride     = 1,
    windows = [
        dict(name='piedemonte_palma',   lonlat=[-73.25, 3.75], size_km=20),
        dict(name='altillanura_gaitan', lonlat=[-72.10, 4.30], size_km=20),
        dict(name='sabana_inundable',   lonlat=[-71.30, 5.60], size_km=20),
    ],
)
QUICK = os.environ.get('HIDRO_QUICK') == '1'
if QUICK:
    CFG.update(epochs=3, patience=2)

PROJ_DIR = Path('/content') if IN_COLAB else Path(os.environ.get('HIDRO_BASE', '.')).resolve()
DATA_DIR = PROJ_DIR / 'data_v2'; DATA4 = PROJ_DIR / 'data_v4'; OUT_DIR = PROJ_DIR / 'results_v6'; MAP_DIR = OUT_DIR / 'maps'
for p in (DATA_DIR, DATA4, OUT_DIR, MAP_DIR): p.mkdir(parents=True, exist_ok=True)
GPKG = PROJ_DIR / CFG['gpkg']
BASE_DATASET = DATA_DIR / 'dataset_meta_v2.npz'; DATASET = DATA4 / 'dataset_meta_v4.npz'

if IN_COLAB:
    import subprocess; subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'rasterio'], check=False)
    from google.colab import files
    missing_files = [p for p in (GPKG, DATASET) if not p.exists() and not (PROJ_DIR / p.name).exists()]
    if missing_files:
        print('Upload:', [p.name for p in missing_files]); up = files.upload()
    for p in (DATASET,):
        if not p.exists() and (PROJ_DIR / p.name).exists(): (PROJ_DIR / p.name).rename(p)

STATE_FILE = OUT_DIR / 'workflow_state.json'
STATE = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {'steps': {}}
REPEATABLE = ('basin', 'gee', 'samples', 'windows', 'borders', 'flags', 'load', 'diagnostics', 'evaluate', 'maps', 'areas', 'final_map')

def retry(fn, tries=6):
    for i in range(tries):
        try:
            return fn()
        except (TypeError, KeyError, ValueError, IndexError):
            raise
        except Exception as e:
            if i == tries - 1: raise
            time.sleep(min(60, 2 ** i + random.random()))

def save_state():
    STATE_FILE.write_text(json.dumps(STATE, indent=1, ensure_ascii=False))

def should_run(step):
    if not RUN.get(step, False):
        print(f'⏭  {step}: disabled in profile {PROFILE}'); return False
    if step in STATE['steps'] and step not in FORCE:
        print(f'✔  {step}: completed on {STATE["steps"][step]}' +
              (' (recomputed / resumed)' if step in REPEATABLE else ' (add it to FORCE to repeat)'))
        return step in REPEATABLE
    return True

def mark_done(step):
    STATE['steps'][step] = datetime.datetime.now().isoformat(timespec='seconds'); save_state()

print('Profile:', PROFILE, '|', {k: v for k, v in RUN.items() if v})
print('Project :', PROJ_DIR); print('Data    :', DATA_DIR); print('Results :', OUT_DIR)
print('Completed steps:', list(STATE['steps']))

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import tensorflow as tf, keras
from keras import layers

S2_FEATS = ['d_B2', 'd_B3', 'd_B4', 'd_B5', 'd_B6', 'd_B7', 'd_B8', 'd_B8A', 'd_B11', 'd_B12',
            'd_NDVI', 'd_MNDWI', 'd_NDMI', 'w_B8', 'w_B11', 'w_NDVI', 'w_MNDWI', 'w_NDMI']
S1_FEATS = ['s1d_VV', 's1d_VH', 's1w_VV', 's1w_VH', 's1w_VVp10', 's1w_VHVV']
FEATURE_BANDS = S2_FEATS + S1_FEATS
BAND_SCALE = np.array([10000.] * len(S2_FEATS) + [100.] * len(S1_FEATS), np.float32)
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

MB_FROM = [3, 49, 6, 11, 82, 12, 13, 81, 50, 21, 74, 35, 9, 23, 24, 25, 29, 30, 32, 68, 75, 33, 31]
MB_TO   = [0, 0,  1, 2,  2,  3,  3,  3,  3,  4,  4,  5,  5, 6,  6,  6,  6,  6,  6,  6,  6,  7,  7]
CLASS_NAMES  = ['Forest', 'Flooded forest', 'Flooded grassland', 'Grassland/savanna', 'Farming',
                'Plantations', 'Non-vegetated', 'Water']
CLASS_COLORS = ['#1F8D49', '#026975', '#519799', '#D6BC74', '#FFEFC3', '#9065D0', '#D4271E', '#2532E4']
HYDRIC = ['Flooded forest', 'Flooded grassland', 'Water']
HYD6_NAMES = ['Forest', 'Flooded forest', 'Flooded grassland', 'Grassland', 'Anthropic', 'Water']
TO6 = np.array([0, 1, 2, 3, 4, 4, 4, 5])
K, K6, C, PATCH = len(CLASS_NAMES), len(HYD6_NAMES), len(FEATURE_BANDS), 2 * CFG['patch_radius'] + 1
NODATA_INT16, LABEL_NODATA = -32768, 255
CMAP = ListedColormap(CLASS_COLORS); NORM = BoundaryNorm(np.arange(K + 1) - .5, K)
LEGEND = [Patch(color=c, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
keras.utils.set_random_seed(CFG['seed'])
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'no (CPU)')
if not tf.config.list_physical_devices('GPU') and not IN_COLAB:
    print('⚠ No GPU: the kernel did not receive LD_LIBRARY_PATH. Select the "Python (hidro_meta_gpu)" kernel registered with '
          'ipykernel (Select Kernel → Jupyter Kernel…), not "Python Environments". Running on CPU.')

## 1 · Meta River Basin (Hydrographic Subzones)

In [ ]:
import geopandas as gpd
if should_run('basin'):
    SZH = gpd.read_file(GPKG).to_crs(4326)
    SZH['area_km2'] = SZH.to_crs(3116).area / 1e6
    SZH['COD_SZH'] = SZH['COD_SZH'].astype(str)
    BASIN = SZH.dissolve()
    print(f"{len(SZH)} subzones · {SZH.area_km2.sum():,.0f} km² · ZH {SZH.NOM_ZH.iloc[0]}")

    from shapely.geometry import Point, box
    WIN_GDF = gpd.GeoDataFrame(
        [{'name': w['name'], 'geometry': Point(w['lonlat'])} for w in CFG['windows']], crs=4326)
    WIN_GDF = WIN_GDF.sjoin(SZH[['NOM_SZH', 'geometry']], how='left').drop(columns='index_right')
    outside = WIN_GDF[WIN_GDF.NOM_SZH.isna()]
    if len(outside): print('⚠ Windows outside the basin:', outside.name.tolist())

    fig, ax = plt.subplots(figsize=(9, 6))
    SZH.plot(ax=ax, column='NOM_SZH', cmap='tab20', edgecolor='k', linewidth=.4, alpha=.6)
    for _, r in SZH.iterrows():
        p = r.geometry.representative_point(); ax.text(p.x, p.y, r.COD_SZH, fontsize=6, ha='center')
    w_box = WIN_GDF.to_crs(CFG['crs']).buffer(10_000, cap_style=3).to_crs(4326)
    w_box.boundary.plot(ax=ax, color='red', linewidth=1.5)
    for _, r in WIN_GDF.iterrows(): ax.annotate(r['name'], r.geometry.coords[0], color='red', fontsize=8)
    ax.set_title('Meta River basin — hydrographic subzones (IDEAM) and mapping windows')
    plt.savefig(OUT_DIR / 'basin_subzones.png', dpi=200, bbox_inches='tight'); plt.show()
    display(SZH[['COD_SZH', 'NOM_SZH', 'area_km2']].sort_values('COD_SZH').round(0).reset_index(drop=True))
    mark_done('basin')

## 2 · Google Earth Engine

In [ ]:
if should_run('gee'):
    import ee
    HV = 'https://earthengine-highvolume.googleapis.com'
    try:
        ee.Initialize(project=CFG['ee_project'], opt_url=HV)
    except Exception:
        ee.Authenticate(); ee.Initialize(project=CFG['ee_project'], opt_url=HV)
    ee.data.setDeadline(180_000)

    from shapely.geometry import mapping
    def to_ee_geom(geom, tol=0.001):
        gj = json.loads(json.dumps(mapping(geom.simplify(tol, preserve_topology=True))))
        return ee.Geometry(gj, None, False)

    AOI = to_ee_geom(BASIN.geometry.iloc[0])
    SZH_FC = ee.FeatureCollection([
        ee.Feature(to_ee_geom(r.geometry), {'COD_SZH': r.COD_SZH, 'NOM_SZH': r.NOM_SZH})
        for _, r in SZH.iterrows()])
    print('AOI area in EE (km²):', round(AOI.area(1000).divide(1e6).getInfo()))

    MB_ID = 'projects/mapbiomas-colombia/assets/LULC/COLECCION3/INTEGRACION/COLOMBIA-1'
    mb_type = ee.data.getAsset(MB_ID)['type'].upper()
    MB = ee.ImageCollection(MB_ID).mosaic() if 'COLLECTION' in mb_type else ee.Image(MB_ID)
    mb_bands = MB.bandNames().getInfo()
    print(f'MapBiomas ({mb_type}):', mb_bands[:3], '…', mb_bands[-3:])
    years_needed = CFG['train_years'] + [CFG['test_year']]
    missing = [y for y in years_needed if f'classification_{y}' not in mb_bands]
    assert not missing, f'Missing classification_ bands for {missing}: check the asset'

    def mb_label(year):
        lab = MB.select(f'classification_{year}').remap(MB_FROM, MB_TO, LABEL_NODATA).rename('label')
        return lab.updateMask(lab.neq(LABEL_NODATA)).toInt16()

    def homogeneous(lab):
        return (lab.reproject(crs='EPSG:4326', scale=30)
                   .reduceNeighborhood(reducer=ee.Reducer.countDistinct(), kernel=ee.Kernel.square(1)).eq(1))

    STABLE = ee.ImageCollection([mb_label(y) for y in CFG['train_years']]) \
               .reduce(ee.Reducer.countDistinct()).eq(1)

    CSP = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    def s2_collection(start, end, roi=None):
        return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(AOI if roi is None else roi).filterDate(start, end)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CFG['max_cloud_scene']))
                .linkCollection(CSP, ['cs_cdf'])
                .map(lambda img: img.updateMask(img.select('cs_cdf').gte(CFG['clear_threshold']))
                                    .select(S2_BANDS).divide(10000)))

    def indices(img, p):
        return ee.Image.cat([img.normalizedDifference(['B8', 'B4']).rename(p + 'NDVI'),
                             img.normalizedDifference(['B3', 'B11']).rename(p + 'MNDWI'),
                             img.normalizedDifference(['B8', 'B11']).rename(p + 'NDMI')])

    def s1_collection(start, end, roi=None):
        return (ee.ImageCollection('COPERNICUS/S1_GRD')
                .filterBounds(AOI if roi is None else roi).filterDate(start, end)
                .filter(ee.Filter.eq('instrumentMode', 'IW'))
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
                .select(['VV', 'VH'])
                .map(lambda img: img.updateMask(img.select('VV').gt(-30))))

    S1_EMPTY = ee.Image.constant([0, 0]).rename(['VV', 'VH']).toFloat().updateMask(ee.Image.constant(0))
    S2_EMPTY = ee.Image.constant([0] * len(S2_BANDS)).rename(S2_BANDS).toFloat().updateMask(ee.Image.constant(0))

    def safe_median(col, empty):
        return ee.Image(ee.Algorithms.If(col.size().gt(0), col.median(), empty))

    def safe_p10(col):
        return ee.Image(ee.Algorithms.If(col.size().gt(0),
                                         col.select('VV').reduce(ee.Reducer.percentile([10])).rename('VV'),
                                         S1_EMPTY.select('VV')))

    def s1_features(year, roi=None):
        D = ee.Date.fromYMD
        dry = s1_collection(D(year - 1, 12, 1), D(year, 4, 1), roi)
        wet = s1_collection(D(year, 5, 1), D(year, 12, 1), roi)
        ann = safe_median(s1_collection(D(year, 1, 1), D(year + 1, 1, 1), roi), S1_EMPTY)
        multi = safe_median(s1_collection(D(year - 1, 1, 1), D(year + 2, 1, 1), roi), S1_EMPTY)
        base = ann.unmask(multi)
        d, w = safe_median(dry, S1_EMPTY).unmask(base), safe_median(wet, S1_EMPTY).unmask(base)
        p10 = safe_p10(wet).unmask(base.select('VV'))
        return ee.Image.cat([d.select(['VV', 'VH'], ['s1d_VV', 's1d_VH']),
                             w.select(['VV', 'VH'], ['s1w_VV', 's1w_VH']),
                             p10.rename('s1w_VVp10'),
                             w.select('VH').subtract(w.select('VV')).rename('s1w_VHVV')])

    def features_for_year(year, roi=None):
        D = ee.Date.fromYMD
        multi = safe_median(s2_collection(D(year - 1, 1, 1), D(year + 2, 1, 1), roi), S2_EMPTY)
        annual = safe_median(s2_collection(D(year, 1, 1), D(year + 1, 1, 1), roi), S2_EMPTY).unmask(multi)
        dry = safe_median(s2_collection(D(year - 1, 12, 1), D(year, 4, 1), roi), S2_EMPTY).unmask(annual)
        wet = safe_median(s2_collection(D(year, 5, 1), D(year, 12, 1), roi), S2_EMPTY).unmask(annual)
        return ee.Image.cat([dry.select(S2_BANDS, ['d_' + b for b in S2_BANDS]), indices(dry, 'd_'),
                             wet.select(['B8', 'B11'], ['w_B8', 'w_B11']), indices(wet, 'w_'),
                             s1_features(year, roi)]).select(FEATURE_BANDS).toFloat()

    from IPython.display import Image as IPImage, display
    region = AOI.bounds()
    try:
        thumb_rgb = features_for_year(CFG['test_year']).select(['d_B4', 'd_B3', 'd_B2']) \
            .visualize(min=0, max=0.15).clip(AOI)
        thumb_mb = mb_label(CFG['test_year']).visualize(min=0, max=K - 1,
            palette=[c[1:] for c in CLASS_COLORS]).clip(AOI)
        thumb_s1 = features_for_year(CFG['test_year']).select(['s1w_VV', 's1w_VH', 's1w_VVp10']) \
            .visualize(min=[-20, -27, -25], max=[-3, -10, -5]).clip(AOI)
        for t, im in [('Sentinel-2 dry season 2024', thumb_rgb), ('Sentinel-1 wet season 2024 (VV, VH, VV p10)', thumb_s1),
                      ('MapBiomas 2024 (8 classes)', thumb_mb)]:
            print(t); display(IPImage(url=im.getThumbURL({'region': region, 'dimensions': 900, 'format': 'png'})))
    except Exception as e:
        print('Preview not available:', e)
    mark_done('gee')

### 2b · MapBiomas Areas and Change Rates by Subzone (2021–2024)

In [ ]:
STATS_CSV = OUT_DIR / 'mapbiomas_areas_subzone.csv'; CHG_CSV = OUT_DIR / 'mapbiomas_change_subzone.csv'
if should_run('gee_stats'):
    yrs = CFG['train_years'] + [CFG['test_year']]
    img = ee.Image.cat([mb_label(y).rename(f'y{y}') for y in yrs])
    px_ha = CFG['stats_scale'] ** 2 / 1e4
    rows = []
    for _, r in SZH.iterrows():
        h = retry(lambda: img.reduceRegion(reducer=ee.Reducer.frequencyHistogram(), geometry=to_ee_geom(r.geometry),
                                           scale=CFG['stats_scale'], maxPixels=1e10, tileScale=8).getInfo())
        for y in yrs:
            for k, n in (h.get(f'y{y}') or {}).items():
                rows.append({'COD_SZH': r.COD_SZH, 'NOM_SZH': r.NOM_SZH, 'year': y,
                             'class': CLASS_NAMES[int(float(k))], 'ha': n * px_ha})
        print('  ', r.NOM_SZH)
    pd.DataFrame(rows).to_csv(STATS_CSV, index=False)

    chg = ee.Image.cat([mb_label(y0).neq(mb_label(y0 + 1)).rename(f'c{y0}') for y0 in yrs[:-1]] +
                       [mb_label(y0).neq(mb_label(y0 + 1)).And(mb_label(y0 + 1).eq(mb_label(y0 + 2))).rename(f'p{y0}')
                        for y0 in yrs[:-2]])
    crow = []
    for _, r in SZH.iterrows():
        m = retry(lambda: chg.reduceRegion(reducer=ee.Reducer.mean(), geometry=to_ee_geom(r.geometry),
                                           scale=CFG['stats_scale'], maxPixels=1e10, tileScale=8).getInfo())
        crow.append({'COD_SZH': r.COD_SZH, 'NOM_SZH': r.NOM_SZH, 'area_km2': r.area_km2,
                     **{f'{y0}-{y0 + 1}': m.get(f'c{y0}') for y0 in yrs[:-1]},
                     **{f'persist_{y0}': m.get(f'p{y0}') for y0 in yrs[:-2]}})
    pd.DataFrame(crow).to_csv(CHG_CSV, index=False)
    mark_done('gee_stats')

if CHG_CSV.exists():
    ch = pd.read_csv(CHG_CSV); per = [c for c in ch.columns if '-' in c]; pcol = [c for c in ch.columns if c.startswith('persist_')]
    ch['gross_rate_%'] = ch[per].mean(axis=1) * 100
    ch['persistent_rate_%'] = ch[pcol].mean(axis=1) * 100 if pcol else np.nan
    for c_ in ['gross_rate_%', 'persistent_rate_%']:
        print(f'MapBiomas {c_} in the basin: {np.average(ch[c_], weights=ch.area_km2):.2f} %/yr')
    display(ch.sort_values('gross_rate_%', ascending=False)[['NOM_SZH', 'gross_rate_%', 'persistent_rate_%']].round(2).head(10))

if STATS_CSV.exists():
    st = pd.read_csv(STATS_CSV)
    last = st[st.year == CFG['test_year']].pivot_table(index='NOM_SZH', columns='class', values='ha', aggfunc='sum').fillna(0)
    last = last[[c for c in CLASS_NAMES if c in last]]
    frac = last.div(last.sum(axis=1), axis=0).sort_values(by=[c for c in HYDRIC if c in last], ascending=False)
    ax = frac.plot.barh(stacked=True, figsize=(10, 7), color=[CLASS_COLORS[CLASS_NAMES.index(c)] for c in frac])
    ax.set_xlabel('Area fraction'); ax.set_title(f'MapBiomas land cover {CFG["test_year"]} by subzone')
    ax.legend(bbox_to_anchor=(1, 1), fontsize=7); plt.savefig(OUT_DIR / 'land_cover_subzone.png', dpi=200, bbox_inches='tight'); plt.show()
    hyd = st[st['class'].isin(HYDRIC)].pivot_table(index='year', columns='class', values='ha', aggfunc='sum')
    display((hyd / 1e3).round(1).rename(columns=lambda c: c + ' (kha)'))

## 3 · Direct Download from Earth Engine

In [ ]:
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
TO_UTM = Transformer.from_crs(4326, CFG['crs'], always_xy=True)
R = CFG['patch_radius']

def grid(x0, y0, w, h):
    return {'dimensions': {'width': int(w), 'height': int(h)},
            'affineTransform': {'scaleX': 10, 'shearX': 0, 'translateX': float(x0),
                                'shearY': 0, 'scaleY': -10, 'translateY': float(y0)},
            'crsCode': CFG['crs']}

def patch_grid(lon, lat):
    x, y = TO_UTM.transform(lon, lat)
    px, py = np.floor(x / 10), np.floor(y / 10)
    return grid((px - R) * 10, (py + 1 + R) * 10, PATCH, PATCH)

def npy_to_array(a, bands):
    if isinstance(a, (bytes, bytearray)):
        a = np.load(io.BytesIO(a), allow_pickle=False)
    return np.stack([a[b] for b in bands], axis=-1)

def sample_points(year):
    out = DATA_DIR / f'puntos_{year}.csv'
    if out.exists(): return pd.read_csv(out)
    is_test = year == CFG['test_year']
    lab = mb_label(year)
    mask = lab.mask() if is_test else homogeneous(lab).And(STABLE)
    base = lab.updateMask(mask).addBands(ee.Image.pixelLonLat())
    frac = (SZH.area_km2 / SZH.area_km2.sum()).values
    dfs = []
    for (_, r), f in zip(SZH.iterrows(), frac):
        n = int(max(10, round(CFG['n_per_class'] * f * 1.3)))
        pts = base.stratifiedSample(numPoints=n, classBand='label', region=to_ee_geom(r.geometry), scale=30,
                                    seed=CFG['seed'] + year, geometries=False, dropNulls=True, tileScale=16)
        df = retry(lambda: ee.data.computeFeatures({'expression': pts.select(['label', 'longitude', 'latitude']),
                                                    'fileFormat': 'PANDAS_DATAFRAME'}))
        df['szh'] = r.NOM_SZH; dfs.append(df)
        print(f'  {year} · {r.NOM_SZH[:35]:35s} {len(df):5d} points')
    df = pd.concat(dfs, ignore_index=True)
    df = df.sample(frac=1, random_state=CFG['seed']).groupby('label').head(CFG['n_per_class'])
    df['year'] = year; df['label'] = df['label'].astype(int)
    df = df.reset_index(drop=True); df['pid'] = np.arange(len(df))
    df.to_csv(out, index=False)
    return df

def fetch_patch(year, lon, lat):
    img = features_for_year(year, ee.Geometry.Point([float(lon), float(lat)]).buffer(150)).unmask(0)
    a = retry(lambda: ee.data.computePixels({'expression': img, 'fileFormat': 'NPY', 'grid': patch_grid(lon, lat)}))
    return npy_to_array(a, FEATURE_BANDS).astype('float32')

def probe_patches(year, pts, n=3):
    print(f'  Testing {n} patches from {year}…', flush=True)
    ts = []
    for r in pts.head(n).itertuples():
        t0 = time.time(); a = fetch_patch(year, r.longitude, r.latitude); ts.append(time.time() - t0)
        assert a.shape == (PATCH, PATCH, C), a.shape
    est = np.mean(ts) * len(pts) / CFG['workers'] / 60
    print(f'  ✔ {np.mean(ts):.1f} s per patch (sequential) · estimated ~{est:.0f} min for {len(pts)} patches '
          f'with {CFG["workers"]} workers', flush=True)

def download_patches(year, pts, pdir=None, tag='parches'):
    pdir = pdir or DATA_DIR / 'parches'; pdir.mkdir(parents=True, exist_ok=True)
    pend = [c0 for c0 in range(0, len(pts), CFG['chunk'])
            if not (pdir / f'{tag}_{year}_{c0 // CFG["chunk"]:04d}.npz').exists()]
    if not pend: print(f'  {year}: all chunks already downloaded'); return
    probe_patches(year, pts)
    print(f'  {year}: {len(pend)} pending chunks of {CFG["chunk"]} patches', flush=True)
    for c0 in range(0, len(pts), CFG['chunk']):
        f = pdir / f'{tag}_{year}_{c0 // CFG["chunk"]:04d}.npz'
        if f.exists(): continue
        sub_ = pts.iloc[c0:c0 + CFG['chunk']]
        X = np.full((len(sub_), PATCH, PATCH, C), np.nan, np.float32)
        t0 = time.time(); nerr = 0
        with ThreadPoolExecutor(CFG['workers']) as ex:
            futs = {ex.submit(fetch_patch, year, r.longitude, r.latitude): i for i, r in enumerate(sub_.itertuples())}
            for j, fu in enumerate(as_completed(futs), 1):
                if j % 100 == 0:
                    print(f'    … {j}/{len(sub_)} patches · {j / (time.time() - t0):.1f}/s · errors {nerr}', flush=True)
                try: X[futs[fu]] = fu.result()
                except Exception as e:
                    nerr += 1
                    if nerr <= 5: print('   ✗', type(e).__name__, str(e)[:150], flush=True)
        fail = np.isnan(X).any(axis=(1, 2, 3)).mean()
        if fail > 0.2:
            raise RuntimeError(f'{fail:.0%} failed patches in chunk {f.name}: check the ✗ errors above')
        np.savez_compressed(f, X=X.astype('float16'), pid=sub_.pid.values)
        dt = time.time() - t0; n_chunks = -(-len(pts) // CFG['chunk']); i_chunk = c0 // CFG['chunk'] + 1
        eta = dt * (n_chunks - i_chunk) / 60
        print(f'  {year}: chunk {i_chunk}/{n_chunks} in {dt:.0f} s · {len(sub_) / dt:.1f} patches/s · '
              f'errors {nerr} · ~{eta:.0f} min left for this year')

def build_dataset():
    Xs, meta = [], []
    for year in CFG['train_years'] + [CFG['test_year']]:
        pts = pd.read_csv(DATA_DIR / f'puntos_{year}.csv').set_index('pid')
        for f in sorted((DATA_DIR / 'parches').glob(f'parches_{year}_*.npz')):
            z = np.load(f); Xs.append(z['X']); meta.append(pts.loc[z['pid']].reset_index())
    X = np.concatenate(Xs); m = pd.concat(meta, ignore_index=True)
    ok = np.isfinite(X).all(axis=(1, 2, 3)) & ~np.all(X == 0, axis=-1).any(axis=(1, 2))
    s1 = [FEATURE_BANDS.index(b) for b in S1_FEATS]
    no_s1 = np.all(X[:, PATCH // 2, PATCH // 2, :][:, s1] == 0, axis=1)
    if no_s1.any():
        print(f'⚠ {no_s1.sum()} patches without Sentinel-1 (not even ±1 year) → discarded')
        print('   by year:', pd.Series(m.year.values[no_s1]).value_counts().sort_index().to_dict())
    ok &= ~no_s1
    np.savez_compressed(BASE_DATASET, X=X[ok], y=m.label.values[ok], year=m.year.values[ok],
                        lon=m.longitude.values[ok], lat=m.latitude.values[ok], szh=m.szh.to_numpy(dtype=str)[ok],
                        pid=m.pid.values[ok])
    print(f'{BASE_DATASET.name}: {ok.sum():,} valid patches ({(~ok).sum()} discarded) · '
          f'{DATASET.stat().st_size / 1e6:.0f} MB')

if should_run('samples'):
    for y in CFG['train_years'] + [CFG['test_year']]:
        pts = sample_points(y)
        print(f'{y}: {len(pts)} points →', pts.label.value_counts().sort_index().to_dict())
        download_patches(y, pts)
    if not BASE_DATASET.exists(): build_dataset()
    else: print('·', BASE_DATASET.name, 'already exists')
    mark_done('samples')

In [ ]:
import rasterio
from rasterio.transform import from_origin

def to_int16(img):
    scale = ee.Image.constant([float(v) for v in BAND_SCALE]).rename(FEATURE_BANDS)
    return img.select(FEATURE_BANDS).multiply(scale).round().unmask(NODATA_INT16).toInt16()

def download_window(w, year):
    out = DATA_DIR / f"ventana_{w['name']}_{year}.tif"
    if out.exists(): print('·', out.name, 'already exists'); return
    cx, cy = TO_UTM.transform(*w['lonlat']); half = w['size_km'] * 500
    x0, y0 = np.floor((cx - half) / 10) * 10, np.ceil((cy + half) / 10) * 10
    W = Hh = int(round(w['size_km'] * 100)); T = CFG['tile']
    roi = ee.Geometry.Rectangle([x0, y0 - Hh * 10, x0 + W * 10, y0], CFG['crs'], False)
    img = to_int16(features_for_year(year, roi))
    bands = list(FEATURE_BANDS)
    if year == CFG['test_year']:
        for yh in CFG['train_years']:
            img = img.addBands(mb_label(yh).unmask(LABEL_NODATA).toInt16().rename(f'mb_{yh}')); bands.append(f'mb_{yh}')
        img = img.addBands(mb_label(year).unmask(LABEL_NODATA).toInt16()); bands.append('label')
    arr = np.full((Hh, W, len(bands)), NODATA_INT16, np.int16)
    jobs = [(ty, tx) for ty in range(0, Hh, T) for tx in range(0, W, T)]
    def get(ty, tx):
        h, wd = min(T, Hh - ty), min(T, W - tx)
        a = retry(lambda: ee.data.computePixels({'expression': img, 'fileFormat': 'NPY',
                                                  'grid': grid(x0 + tx * 10, y0 - ty * 10, wd, h)}))
        return ty, tx, npy_to_array(a, bands)
    t0 = time.time()
    with ThreadPoolExecutor(min(8, CFG['workers'])) as ex:
        for ty, tx, a in ex.map(lambda j: get(*j), jobs):
            arr[ty:ty + a.shape[0], tx:tx + a.shape[1]] = a
    prof = dict(driver='GTiff', height=Hh, width=W, count=len(bands), dtype='int16', crs=CFG['crs'],
                transform=from_origin(x0, y0, 10, 10), compress='deflate', tiled=True, nodata=NODATA_INT16)
    with rasterio.open(out, 'w', **prof) as dst:
        dst.write(np.moveaxis(arr, -1, 0)); dst.descriptions = tuple(bands)
    print(f'✔ {out.name}: {Hh}×{W}×{len(bands)} in {time.time() - t0:.0f} s')

if should_run('windows'):
    for w in CFG['windows']:
        for y in [CFG['test_year'], CFG['pred_year']]:
            download_window(w, y)
    mark_done('windows')

### 3b · Border Points for Training

In [ ]:
N_BORDER = int(round(CFG['n_per_class'] * CFG['border_frac'] / (1 - CFG['border_frac'])))

def sample_border_points(year):
    out = DATA4 / f'puntos_borde_{year}.csv'
    if out.exists(): return pd.read_csv(out)
    lab = mb_label(year)
    base = lab.updateMask(homogeneous(lab).Not().And(STABLE)).addBands(ee.Image.pixelLonLat())
    frac = (SZH.area_km2 / SZH.area_km2.sum()).values; dfs = []
    for (_, r), f in zip(SZH.iterrows(), frac):
        n = int(max(5, round(N_BORDER * f * 1.3)))
        pts = base.stratifiedSample(numPoints=n, classBand='label', region=to_ee_geom(r.geometry), scale=30,
                                    seed=CFG['seed'] + 7 * year, geometries=False, dropNulls=True, tileScale=16)
        df = retry(lambda: ee.data.computeFeatures({'expression': pts.select(['label', 'longitude', 'latitude']),
                                                    'fileFormat': 'PANDAS_DATAFRAME'}))
        df['szh'] = r.NOM_SZH; dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    df = df.sample(frac=1, random_state=CFG['seed']).groupby('label').head(N_BORDER)
    df['year'] = year; df['label'] = df['label'].astype(int)
    df = df.reset_index(drop=True); df['pid'] = 100_000 + np.arange(len(df))
    df.to_csv(out, index=False); return df

def build_dataset_v4():
    z = np.load(BASE_DATASET, allow_pickle=False); base = {k: z[k] for k in z.files}
    nb = len(base['y']); base.setdefault('pid', np.full(nb, -1)); base['borde'] = np.zeros(nb, np.int8)
    Xs, meta = [], []
    for yy in CFG['train_years']:
        pts = pd.read_csv(DATA4 / f'puntos_borde_{yy}.csv').set_index('pid')
        for f in sorted((DATA4 / 'parches_borde').glob(f'borde_{yy}_*.npz')):
            zz = np.load(f); Xs.append(zz['X']); meta.append(pts.loc[zz['pid']].reset_index())
    X = np.concatenate(Xs); m = pd.concat(meta, ignore_index=True)
    ok = np.isfinite(X).all(axis=(1, 2, 3)) & ~np.all(X == 0, axis=-1).any(axis=(1, 2))
    s1 = [FEATURE_BANDS.index(b) for b in S1_FEATS]
    ok &= ~np.all(X[:, PATCH // 2, PATCH // 2, :][:, s1] == 0, axis=1)
    new = dict(X=X[ok], y=m.label.values[ok], year=m.year.values[ok], lon=m.longitude.values[ok], lat=m.latitude.values[ok],
               szh=m.szh.to_numpy(dtype=str)[ok], pid=m.pid.values[ok], borde=np.ones(ok.sum(), np.int8))
    out = {k: np.concatenate([base[k].astype(new[k].dtype) if k != 'szh' else base[k].astype(str), new[k]]) for k in new}
    np.savez_compressed(DATASET, **out)
    print(f'{DATASET.name}: {nb:,} base + {ok.sum():,} border = {len(out["y"]):,} patches · {DATASET.stat().st_size / 1e6:.0f} MB')

if should_run('borders'):
    print(f'{N_BORDER} border points per class and year')
    for yy in CFG['train_years']:
        pts = sample_border_points(yy)
        print(f'{yy}: {len(pts)} border points →', pts.label.value_counts().sort_index().to_dict())
        download_patches(yy, pts, pdir=DATA4 / 'parches_borde', tag='borde')
    if not DATASET.exists(): build_dataset_v4()
    else: print('·', DATASET.name, 'already exists')
    mark_done('borders')

### 3c · Quality Flags for Test-Year Points

In [ ]:
FLAGS_CSV = DATA_DIR / f"flags_{CFG['test_year']}.csv"
if should_run('flags'):
    if FLAGS_CSV.exists():
        print('·', FLAGS_CSV.name, 'already exists')
    else:
        tyr = CFG['test_year']
        stable_all = ee.ImageCollection([mb_label(yy) for yy in CFG['train_years'] + [tyr]]) \
                       .reduce(ee.Reducer.countDistinct()).eq(1)
        fimg = homogeneous(mb_label(tyr)).rename('homog').addBands(stable_all.rename('estable')).unmask(0)
        pts = pd.read_csv(DATA_DIR / f'puntos_{tyr}.csv'); out = []
        for c0 in range(0, len(pts), 1000):
            sub_ = pts.iloc[c0:c0 + 1000]
            fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r.longitude), float(r.latitude)]), {'pid': int(r.pid)})
                                       for r in sub_.itertuples()])
            red = fimg.reduceRegions(collection=fc, reducer=ee.Reducer.first(), scale=30, tileScale=4)
            out.append(retry(lambda: ee.data.computeFeatures({'expression': red.select(['pid', 'homog', 'estable']),
                                                                'fileFormat': 'PANDAS_DATAFRAME'})))
            print(f'  {c0 + len(sub_)}/{len(pts)} points', flush=True)
        fl = pd.concat(out, ignore_index=True)[['pid', 'homog', 'estable']].merge(
            pts[['pid', 'longitude', 'latitude', 'label']], on='pid')
        fl.to_csv(FLAGS_CSV, index=False)
    fl = pd.read_csv(FLAGS_CSV)
    print(f"Points {CFG['test_year']}: homogeneous {fl.homog.mean():.0%} · stable 2021–24 {fl.estable.mean():.0%} · "
          f"both {(fl.homog * fl.estable).mean():.0%}")
    display((fl.assign(comparable=fl.homog * fl.estable).groupby('label').comparable.mean()
               .rename(index=dict(enumerate(CLASS_NAMES))).to_frame('comparable fraction').round(2)))
    mark_done('flags')

## 4 · Data Loading and Spatio-Temporal Split

In [ ]:
def load_dataset(path=DATASET):
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} not found: run step 3 (profile pc_download) or upload it to Colab')
    z = np.load(path, allow_pickle=False)
    d = {k: z[k] for k in z.files}
    d['X'] = d['X'].astype('float32'); d['y'] = d['y'].astype(int); d['year'] = d['year'].astype(int)
    print(f"{path.name}: {len(d['y']):,} patches · {d['X'].shape}")
    return d

import hashlib
def spatial_blocks(lon, lat, block_deg, fractions=(0.6, 0.2, 0.2), seed=42, buffer_deg=0.002):
    bx, by = np.floor(lon / block_deg).astype(int), np.floor(lat / block_deg).astype(int)
    keys = np.char.add(np.char.add(bx.astype(str), '_'), by.astype(str))
    u = {k: int(hashlib.md5(f'{seed}_{k}'.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF for k in np.unique(keys)}
    role = {k: 'train' if v < fractions[0] else ('val' if v < fractions[0] + fractions[1] else 'test') for k, v in u.items()}
    split = np.array([role[k] for k in keys], dtype=object)
    fx, fy, b = lon / block_deg - bx, lat / block_deg - by, buffer_deg / block_deg
    split[(fx < b) | (fx > 1 - b) | (fy < b) | (fy > 1 - b)] = 'buffer'
    return split

if should_run('load'):
    if not DATASET.exists():
        raise FileNotFoundError(f'{DATASET} is missing: run profile pc_full (download) once before pc_final')
    D = load_dataset()
    split = spatial_blocks(D['lon'], D['lat'], CFG['block_deg'], seed=CFG['seed'])
    ty, tt = np.isin(D['year'], CFG['train_years']), D['year'] == CFG['test_year']
    S = {'train': ty & (split == 'train'), 'val': ty & (split == 'val'), 'test_S': ty & (split == 'test'),
         'test_T': tt & (split == 'train'), 'test_ST': tt & (split == 'test')}
    S = {k: np.where(v)[0] for k, v in S.items()}

    BORDER = D.get('borde', np.zeros(len(D['y']), np.int8)).astype(bool)
    COMP = np.ones(len(D['y']), bool); FLAG_H = ~BORDER; FLAG_E = np.ones(len(D['y']), bool)
    if FLAGS_CSV.exists():
        fl = pd.read_csv(FLAGS_CSV)
        key = lambda lo, la: np.char.add(np.char.add(np.round(lo, 6).astype(str), '_'), np.round(la, 6).astype(str))
        kk = key(fl.longitude.values, fl.latitude.values)
        h_map, e_map = dict(zip(kk, fl.homog.fillna(0) > 0)), dict(zip(kk, fl.estable.fillna(0) > 0))
        is_ty = D['year'] == CFG['test_year']; kd = key(D['lon'][is_ty], D['lat'][is_ty])
        FLAG_H[is_ty] = [h_map.get(k_, False) for k_ in kd]; FLAG_E[is_ty] = [e_map.get(k_, False) for k_ in kd]
        print(f"2024 comparable: {COMP[is_ty].sum():,} of {is_ty.sum():,} ({COMP[is_ty].mean():.0%})")
    else:
        print('⚠ No 2024 flags (step 3c): test_*_c = full test_*')
    COMP = FLAG_H & FLAG_E
    print(f"Border points: train {BORDER[S['train']].mean():.0%} · val {BORDER[S['val']].mean():.0%} · test_S {BORDER[S['test_S']].mean():.0%}")
    S['test_T_c'], S['test_ST_c'] = S['test_T'][COMP[S['test_T']]], S['test_ST'][COMP[S['test_ST']]]
    S['test_S_c'] = S['test_S'][COMP[S['test_S']]]


    display(pd.crosstab(pd.Series(D['y']).map(dict(enumerate(CLASS_NAMES))), D['year'], margins=True))
    print({k: len(v) for k, v in S.items()}, '| buffer:', int((split == 'buffer').sum()))

    fig, ax = plt.subplots(figsize=(8, 6))
    SZH.boundary.plot(ax=ax, color='grey', linewidth=.4)
    for k, c in {'train': '#1f77b4', 'val': '#ff7f0e', 'test': '#d62728', 'buffer': '#bbbbbb'}.items():
        m = split == k; ax.scatter(D['lon'][m], D['lat'][m], s=1.5, c=c, label=k)
    ax.legend(markerscale=6); ax.set_title('Spatial block split')
    plt.savefig(OUT_DIR / 'spatial_split.png', dpi=200, bbox_inches='tight'); plt.show()

    c = D['X'][S['train']].reshape(-1, C)
    MEAN, STD = c.mean(0).astype('float32'), (c.std(0) + 1e-6).astype('float32')
    json.dump({'mean': MEAN.tolist(), 'std': STD.tolist(), 'bands': FEATURE_BANDS},
              open(OUT_DIR / 'normalization.json', 'w'), indent=1)
    Xn = ((D['X'] - MEAN) / STD).astype('float32'); y = D['y']
    cnt = np.bincount(y[S['train']], minlength=K).astype(float)
    cw = np.where(cnt > 0, cnt.sum() / ((cnt > 0).sum() * np.maximum(cnt, 1)), 0.0)
    CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
    mark_done('load')

### 4b · Test-Year Diagnostics

In [ ]:
DIAG_VARS = ['d_NDVI', 'd_MNDWI', 'd_NDMI', 'w_NDVI', 'w_MNDWI', 's1d_VV', 's1w_VV', 's1w_VVp10']
if should_run('diagnostics'):
    tyr, cc = CFG['test_year'], PATCH // 2
    sel = COMP
    dd = pd.DataFrame(D['X'][sel, cc, cc][:, [FEATURE_BANDS.index(v) for v in DIAG_VARS]], columns=DIAG_VARS)
    dd['year'], dd['cls'] = D['year'][sel], np.array(CLASS_NAMES)[D['y'][sel]]

    fig, axs = plt.subplots(2, 4, figsize=(16, 6.5), sharex=True)
    for ax, v in zip(axs.ravel(), DIAG_VARS):
        med = dd.groupby(['cls', 'year'])[v].median().unstack(0)
        for cl in CLASS_NAMES:
            if cl in med: ax.plot(med.index, med[cl], marker='o', color=CLASS_COLORS[CLASS_NAMES.index(cl)], label=cl)
        ax.axvspan(tyr - .4, tyr + .4, color='orange', alpha=.15); ax.set_title(v); ax.set_xticks(med.index)
    axs[0, 0].legend(fontsize=6, ncol=2); plt.suptitle('Median by year and class (2024 shaded)')
    plt.tight_layout(); plt.savefig(OUT_DIR / 'diagnostics_annual_medians.png', dpi=200); plt.show()

    eff = []
    for cl in CLASS_NAMES:
        a_, b_ = dd[(dd.cls == cl) & (dd.year != tyr)], dd[(dd.cls == cl) & (dd.year == tyr)]
        if len(b_) < 20: continue
        eff.append({'cls': cl, **{v: (b_[v].mean() - a_[v].mean()) / np.sqrt((a_[v].var() + b_[v].var()) / 2)
                                  for v in DIAG_VARS}})
    eff = pd.DataFrame(eff).set_index('cls'); eff.to_csv(OUT_DIR / 'diagnostics_effect_2024.csv')
    print("Effect size (Cohen's d) 2024 vs 2021–23 · red = 2024 higher, blue = lower")
    display(eff.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1, axis=None).format('{:.2f}'))

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_predict
    from sklearn.metrics import roc_auc_score
    auc = []
    Fc = D['X'][sel, cc, cc]
    for k_, cl in enumerate(CLASS_NAMES):
        m_ = D['y'][sel] == k_; yy = (D['year'][sel][m_] == tyr).astype(int)
        if yy.sum() < 30 or (1 - yy).sum() < 30: continue
        pr = cross_val_predict(RandomForestClassifier(200, min_samples_leaf=5, class_weight='balanced', n_jobs=-1,
                                                      random_state=CFG['seed']), Fc[m_], yy, cv=5, method='predict_proba')[:, 1]
        auc.append({'cls': cl, 'n_2024': int(yy.sum()), 'AUC_is_2024': roc_auc_score(yy, pr)})
    auc = pd.DataFrame(auc); auc.to_csv(OUT_DIR / 'diagnostics_auc_2024.csv', index=False)
    print('AUC of the "is it 2024?" classifier by class (0.5 = no shift)'); display(auc.round(3))

    if 'ee' in globals() and 'AOI' in globals():
      try:
        ch_ = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')
        def rainfall(yv):
            yv = ee.Number(yv)
            dry = ch_.filterDate(ee.Date.fromYMD(yv.subtract(1), 12, 1), ee.Date.fromYMD(yv, 4, 1)).sum().rename('dry')
            annual = ch_.filterDate(ee.Date.fromYMD(yv, 1, 1), ee.Date.fromYMD(yv.add(1), 1, 1)).sum().rename('annual')
            r_ = dry.addBands(annual).reduceRegion(ee.Reducer.mean(), AOI, 5000, maxPixels=1e9, tileScale=4)
            return ee.Feature(None, r_).set('year', yv)
        if True:
            rr_ = retry(lambda: ee.FeatureCollection(ee.List.sequence(2001, CFG['pred_year']).map(rainfall)).getInfo())
            pp = pd.DataFrame([f['properties'] for f in rr_['features']]).set_index('year').sort_index()
            ref = pp.loc[2001:2020].mean()
            anom = (pp / ref - 1) * 100; anom.to_csv(OUT_DIR / 'diagnostics_chirps_anomaly.csv')
            fig, ax = plt.subplots(figsize=(9, 3.5)); yrs_ = anom.loc[2015:].index
            ax.bar(yrs_ - .2, anom.loc[yrs_, 'dry'], .4, label='Dry season (Dec–Mar)')
            ax.bar(yrs_ + .2, anom.loc[yrs_, 'annual'], .4, label='Year')
            ax.axhline(0, color='k', lw=.8); ax.set_ylabel('Rainfall anomaly (%)'); ax.set_xticks(yrs_)
            ax.set_title('CHIRPS rainfall in the Meta basin vs. 2001–2020'); ax.legend()
            plt.tight_layout(); plt.savefig(OUT_DIR / 'diagnostics_chirps.png', dpi=200); plt.show()
            display(anom.loc[2019:].round(1))
      except Exception as e:
        print('CHIRPS not available:', e)
    mark_done('diagnostics')

## 5 · Models

In [ ]:
@keras.saving.register_keras_serializable(package='hidro')
class ClassToken(layers.Layer):
    def build(self, s): self.cls = self.add_weight(name='cls', shape=(1, 1, s[-1]), initializer='zeros')
    def call(self, x): return tf.concat([tf.tile(self.cls, [tf.shape(x)[0], 1, 1]), x], axis=1)

@keras.saving.register_keras_serializable(package='hidro')
class AddPositionEmbedding(layers.Layer):
    def build(self, s): self.pos = self.add_weight(name='pos', shape=(1, s[1], s[2]),
                                                   initializer=keras.initializers.RandomNormal(stddev=.02))
    def call(self, x): return x + self.pos

@keras.saving.register_keras_serializable(package='hidro')
class TakeToken(layers.Layer):
    def call(self, x): return x[:, 0]

def transformer_block(x, dim, heads, mlp, drop=0.1):
    h = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Add()([x, layers.MultiHeadAttention(heads, dim // heads, dropout=drop)(h, h)])
    h = layers.LayerNormalization(epsilon=1e-6)(x)
    h = layers.Dropout(drop)(layers.Dense(dim)(layers.Dropout(drop)(layers.Dense(mlp, activation='gelu')(h))))
    return layers.Add()([x, h])

def build_cnn3d2d():
    inp = layers.Input((PATCH, PATCH, C)); x = layers.Reshape((PATCH, PATCH, C, 1))(inp)
    for f, k in [(8, 7), (16, 5), (32, 3)]: x = layers.Conv3D(f, (3, 3, k), activation='relu')(x)
    s = x.shape; x = layers.Reshape((s[1], s[2], s[3] * s[4]))(x)
    x = layers.BatchNormalization()(layers.Conv2D(64, 3, activation='relu')(x))
    x = layers.Dropout(.4)(layers.Dense(128, activation='relu')(layers.Flatten()(x)))
    x = layers.Dropout(.4)(layers.Dense(64, activation='relu')(x))
    return keras.Model(inp, layers.Dense(K, activation='softmax')(x), name='CNN3D2D')

def build_vit(sub=3, dim=64, depth=4, heads=4, mlp=128):
    g = PATCH // sub; inp = layers.Input((PATCH, PATCH, C))
    x = layers.Reshape((g, sub, g, sub, C))(inp); x = layers.Permute((1, 3, 2, 4, 5))(x)
    x = layers.Dense(dim)(layers.Reshape((g * g, sub * sub * C))(x))
    x = layers.Dropout(.1)(AddPositionEmbedding()(ClassToken()(x)))
    for _ in range(depth): x = transformer_block(x, dim, heads, mlp)
    x = TakeToken()(layers.LayerNormalization(epsilon=1e-6)(x))
    x = layers.Dropout(.3)(layers.Dense(128, activation='gelu')(x))
    return keras.Model(inp, layers.Dense(K, activation='softmax')(x), name='ViT')

def build_hybrid(dim=64, depth=2, heads=4, mlp=128):
    inp = layers.Input((PATCH, PATCH, C)); x = layers.Reshape((PATCH, PATCH, C, 1))(inp)
    x = layers.Conv3D(8, (3, 3, 7), activation='relu')(x); s = x.shape
    x = layers.Reshape((s[1], s[2], s[3] * s[4]))(x)
    x = layers.Conv2D(dim, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(dim, 3, strides=2, padding='same', activation='relu')(x); s = x.shape
    x = AddPositionEmbedding()(layers.Reshape((s[1] * s[2], dim))(x))
    for _ in range(depth): x = transformer_block(x, dim, heads, mlp)
    x = layers.GlobalAveragePooling1D()(layers.LayerNormalization(epsilon=1e-6)(x))
    x = layers.Dropout(.3)(layers.Dense(128, activation='relu')(x))
    return keras.Model(inp, layers.Dense(K, activation='softmax')(x), name='Hybrid')

BUILDERS = {'CNN3D2D': build_cnn3d2d, 'ViT': build_vit, 'Hybrid': build_hybrid}

def rf_features(X):
    c = X.shape[1] // 2
    return np.concatenate([X[:, c, c], X[:, c-1:c+2, c-1:c+2].mean((1, 2)), X.mean((1, 2)), X.std((1, 2))], 1)

def augment(x, lab):
    x = tf.image.rot90(x, tf.random.uniform([], 0, 4, tf.int32))
    return tf.cond(tf.random.uniform([]) > .5, lambda: tf.image.flip_left_right(x), lambda: x), lab

def dataset(X, lab, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, lab))
    if training: ds = ds.shuffle(min(len(X), 50000), seed=CFG['seed']).map(augment, tf.data.AUTOTUNE)
    return ds.batch(CFG['batch']).prefetch(tf.data.AUTOTUNE)

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier
MODELS, HIST = {}, {}
SRC = PROJ_DIR / CFG['model_src']
if CFG['model_src'] and RUN.get('train') and 'train' not in STATE['steps'] and 'train' not in FORCE and (SRC / 'RF.joblib').exists():
    import shutil
    for f in ['RF.joblib', 'CNN3D2D.keras', 'ViT.keras', 'Hybrid.keras', 'times.json', 'normalization.json',
              'log_CNN3D2D.csv', 'log_ViT.csv', 'log_Hybrid.csv']:
        if (SRC / f).exists() and not (OUT_DIR / f).exists(): shutil.copy2(SRC / f, OUT_DIR / f)
    mark_done('train'); print(f'Models reused from {SRC.name}/ (add "train" to FORCE to retrain)')
TIMES = json.loads((OUT_DIR / 'times.json').read_text()) if (OUT_DIR / 'times.json').exists() else {}

if should_run('train'):
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=100 if QUICK else 400, min_samples_leaf=2, class_weight='balanced',
                                n_jobs=-1, random_state=CFG['seed']).fit(rf_features(Xn[S['train']]), y[S['train']])
    joblib.dump(rf, OUT_DIR / 'RF.joblib'); TIMES['RF'] = time.time() - t0; MODELS['RF'] = rf
    print(f"RF: {TIMES['RF']:.0f} s")
    ds_tr, ds_va = dataset(Xn[S['train']], y[S['train']], True), dataset(Xn[S['val']], y[S['val']])
    for name, build in BUILDERS.items():
        keras.utils.set_random_seed(CFG['seed'])
        m = build(); m.compile(optimizer=keras.optimizers.Adam(CFG['lr']), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        print(f'\n=== {name}: {m.count_params():,} parameters ===')
        t0 = time.time()
        m.fit(ds_tr, validation_data=ds_va, epochs=CFG['epochs'], class_weight=CLASS_WEIGHT, verbose=2,
              callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=CFG['patience'], restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau('val_loss', factor=.5, patience=max(1, CFG['patience'] // 2)),
                         keras.callbacks.CSVLogger(str(OUT_DIR / f'log_{name}.csv'))])
        TIMES[name] = time.time() - t0; m.save(OUT_DIR / f'{name}.keras'); MODELS[name] = m
    (OUT_DIR / 'times.json').write_text(json.dumps(TIMES, indent=1))
    mark_done('train')

if 'RF' not in MODELS and (OUT_DIR / 'RF.joblib').exists(): MODELS['RF'] = joblib.load(OUT_DIR / 'RF.joblib')
for n in BUILDERS:
    if n not in MODELS and (OUT_DIR / f'{n}.keras').exists(): MODELS[n] = keras.models.load_model(OUT_DIR / f'{n}.keras')
print('Available models:', list(MODELS))

logs = {n: pd.read_csv(OUT_DIR / f'log_{n}.csv') for n in BUILDERS if (OUT_DIR / f'log_{n}.csv').exists()}
if logs:
    fig, axs = plt.subplots(1, len(logs), figsize=(5 * len(logs), 3.4), sharey=True)
    for ax, (n, h) in zip(np.atleast_1d(axs), logs.items()):
        for col, ls in [('loss', '-'), ('val_loss', '-'), ('accuracy', '--'), ('val_accuracy', '--')]:
            ax.plot(h[col], ls, label=col)
        ax.set_title(n); ax.set_xlabel('Epoch')
    np.atleast_1d(axs)[0].legend(fontsize=7); plt.tight_layout()
    plt.savefig(OUT_DIR / 'training_curves.png', dpi=200); plt.show()

## 6 · Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, confusion_matrix
from scipy.stats import binomtest, chi2
from scipy.optimize import minimize_scalar
from itertools import combinations

CAL_FILE = OUT_DIR / 'calibration.json'
CAL = json.loads(CAL_FILE.read_text()) if CAL_FILE.exists() else {}
ENSEMBLES = {k_: v['members'] for k_, v in CAL.items() if isinstance(v, dict) and 'members' in v}

def predict_keras(m, X):
    bs = CFG['pred_batch']
    while True:
        try:
            return m.predict(X, batch_size=bs, verbose=0)
        except (tf.errors.InternalError, tf.errors.ResourceExhaustedError):
            if bs > 32:
                bs //= 4; print(f'   ⚠ GPU out of memory/configuration error: retrying with batches of {bs}')
            else:
                print('   ⚠ predicting on CPU')
                with tf.device('/CPU:0'): return m.predict(X, batch_size=CFG['pred_batch'], verbose=0)

def raw_proba(name, Xnorm):
    m = MODELS[name]
    if name == 'RF':
        p = m.predict_proba(rf_features(Xnorm)); full = np.zeros((len(Xnorm), K), np.float32)
        full[:, m.classes_.astype(int)] = p; return full
    return predict_keras(m, Xnorm)

def calibrate(p, T):
    q = np.power(np.clip(p, 1e-7, 1), 1.0 / T); return (q / q.sum(-1, keepdims=True)).astype(np.float32)

def fit_T(p, t):
    nll = lambda T: -np.mean(np.log(np.clip(calibrate(p, T)[np.arange(len(t)), t], 1e-9, 1)))
    return float(minimize_scalar(nll, bounds=(0.05, 10), method='bounded').x)

def ece(p, t, bins=15):
    conf, pred = p.max(1), p.argmax(1); e = 0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m_ = (conf > lo) & (conf <= hi)
        if m_.any(): e += m_.mean() * abs((pred[m_] == t[m_]).mean() - conf[m_].mean())
    return float(e)

def conf_threshold(p, t, target, min_n=50):
    conf, ok = p.max(1), p.argmax(1) == t
    for th in np.round(np.arange(0.30, 0.995, 0.01), 2):
        m_ = conf >= th
        if m_.sum() >= min_n and ok[m_].mean() >= target: return float(th), float(m_.mean()), float(ok[m_].mean())
    return 0.99, float((conf >= .99).mean()), float(ok[conf >= .99].mean()) if (conf >= .99).any() else np.nan

def probs_all(name, Xnorm):
    if name in CAL and 'members' in CAL[name]:
        p = np.mean([probs_all(m, Xnorm) for m in CAL[name]['members']], axis=0)
        return calibrate(p, CAL[name]['T'])
    p = raw_proba(name, Xnorm)
    return calibrate(p, CAL[name]['T']) if name in CAL else p

def mets(t, p, k=K):
    L = list(range(k))
    return dict(OA=accuracy_score(t, p), Kappa=cohen_kappa_score(t, p, labels=L),
                F1_macro=f1_score(t, p, labels=L, average='macro', zero_division=0))

def boot_ci(t, p, k, n=500, seed=0):
    rng = np.random.default_rng(seed); oa, f1 = [], []
    for idx in rng.integers(0, len(t), (n, len(t))):
        oa.append(accuracy_score(t[idx], p[idx]))
        f1.append(f1_score(t[idx], p[idx], labels=list(range(k)), average='macro', zero_division=0))
    return np.percentile(oa, [2.5, 97.5]), np.percentile(f1, [2.5, 97.5])

def mcnemar(t, p1, p2):
    a1, a2 = p1 == t, p2 == t
    b, c = int((a1 & ~a2).sum()), int((~a1 & a2).sum())
    if b + c == 0: return b, c, 1.0
    if b + c < 25: return b, c, binomtest(b, b + c, .5).pvalue
    return b, c, float(chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1, run)
    return adj

def plot_cm(PR, t, names_cls, k, cj, title, fname, map_fn=lambda a: a):
    names = list(PR)
    fig, axs = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4.3))
    for ax, n in zip(np.atleast_1d(axs), names):
        cm = confusion_matrix(map_fn(t), map_fn(PR[n][cj]), labels=range(k), normalize='true')
        ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
        for i in range(k):
            for j in range(k):
                if cm[i, j] >= .05: ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', fontsize=6,
                                            color='white' if cm[i, j] > .5 else 'black')
        ax.set_xticks(range(k)); ax.set_xticklabels(names_cls, rotation=90, fontsize=7)
        ax.set_yticks(range(k)); ax.set_yticklabels(names_cls if ax is np.atleast_1d(axs)[0] else [], fontsize=7)
        ax.set_title(f'{n} · {title}', fontsize=9)
    plt.tight_layout(); plt.savefig(OUT_DIR / fname, dpi=200); plt.show()

if should_run('evaluate'):
    base = list(MODELS); yv = y[S['val']]
    RAW = {n: {k_: raw_proba(n, Xn[idx]) for k_, idx in S.items() if len(idx) and k_ != 'train'} for n in base}

    CAL = {}
    for n in base:
        T = fit_T(RAW[n]['val'], yv)
        CAL[n] = {'T': T, 'ECE_before': ece(RAW[n]['val'], yv), 'ECE_after': ece(calibrate(RAW[n]['val'], T), yv)}
    PROB = {n: {k_: calibrate(p, CAL[n]['T']) for k_, p in RAW[n].items()} for n in base}
    f1v = {n: f1_score(yv, PROB[n]['val'].argmax(1), average='macro') for n in base}
    top = sorted(f1v, key=f1v.get, reverse=True)
    ENS = top[:3] if CFG['ens_members'] == 'auto' else [m for m in CFG['ens_members'] if m in base]
    print('Validation macro F1:', {k_: round(v, 3) for k_, v in f1v.items()}, '→ Ensemble members:', ENS)
    ENSEMBLES = {'Ensemble': ENS, 'Ensemble4': base}
    for en, mem in ENSEMBLES.items():
        if len(mem) < 2: continue
        pe = {k_: np.mean([PROB[m][k_] for m in mem], axis=0) for k_ in PROB[mem[0]]}
        Te = fit_T(pe['val'], yv)
        CAL[en] = {'T': Te, 'ECE_before': ece(pe['val'], yv), 'ECE_after': ece(calibrate(pe['val'], Te), yv), 'members': mem}
        PROB[en] = {k_: calibrate(p, Te) for k_, p in pe.items()}
    for n in PROB:
        th, cov, prec = conf_threshold(PROB[n]['val'], yv, CFG['target_precision'])
        CAL[n].update(threshold=th, coverage_val=cov, precision_val=prec)
    CAL_FILE.write_text(json.dumps(CAL, indent=1))
    cal_df = pd.DataFrame(CAL).T[['T', 'ECE_before', 'ECE_after', 'threshold', 'coverage_val', 'precision_val']]
    print(f"Calibration and threshold for precision ≥ {CFG['target_precision']:.0%} on validation"); display(cal_df.astype(float).round(3))

    fig, ax = plt.subplots(figsize=(4.6, 4.2)); bins = np.linspace(0, 1, 11)
    for n in PROB:
        pT, tT = PROB[n]['test_T'], y[S['test_T']]; cf, okk = pT.max(1), pT.argmax(1) == tT
        xb = [cf[(cf > lo) & (cf <= hi)].mean() if ((cf > lo) & (cf <= hi)).sum() > 20 else np.nan for lo, hi in zip(bins[:-1], bins[1:])]
        yb = [okk[(cf > lo) & (cf <= hi)].mean() if ((cf > lo) & (cf <= hi)).sum() > 20 else np.nan for lo, hi in zip(bins[:-1], bins[1:])]
        ax.plot(xb, yb, marker='o', label=n)
    ax.plot([0, 1], [0, 1], 'k--', lw=.8); ax.set_xlabel('Calibrated confidence'); ax.set_ylabel('Observed accuracy')
    ax.set_title('Reliability on test_T (2024)'); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUT_DIR / 'reliability_test_T.png', dpi=200); plt.show()

    PRED = {n: {k_: p.argmax(1) for k_, p in P.items()} for n, P in PROB.items()}
    names = list(PRED)
    LEGENDS = {'8 classes': (CLASS_NAMES, K, lambda a: a), '6 hydric': (HYD6_NAMES, K6, lambda a: TO6[a])}
    CJ = ['val', 'test_S', 'test_S_c', 'test_T', 'test_T_c', 'test_ST', 'test_ST_c']

    rows = []
    for leg, (cn, k, f) in LEGENDS.items():
        for n, P in PRED.items():
            for cj in CJ:
                if cj in P and len(S[cj]): rows.append({'Legend': leg, 'Model': n, 'Set': cj, 'n': len(S[cj]),
                                                        **mets(f(y[S[cj]]), f(P[cj]), k)})
    res = pd.DataFrame(rows); res.to_csv(OUT_DIR / 'metrics.csv', index=False)
    print('Macro F1'); display(res.pivot_table(index=['Legend', 'Model'], columns='Set', values='F1_macro')[CJ].round(3))

    dec = res[res.Legend == '8 classes'].pivot(index='Model', columns='Set', values='F1_macro')
    dec = pd.DataFrame({'test_S_c': dec.test_S_c, 'test_ST_c': dec.test_ST_c, 'test_ST': dec.test_ST,
                        'Year effect (S_c→ST_c)': dec.test_ST_c - dec.test_S_c,
                        'Difficulty effect (ST_c→ST)': dec.test_ST - dec.test_ST_c})
    dec.to_csv(OUT_DIR / 'drop_decomposition.csv')
    print('Macro F1 drop decomposition (8 classes, same test blocks)'); display(dec.round(3))

    i24 = np.concatenate([S['test_T'], S['test_ST']]); est = []
    for hh in (True, False):
        for ee_ in (True, False):
            m_ = (FLAG_H[i24] == hh) & (FLAG_E[i24] == ee_)
            if m_.sum() < 30: continue
            est.append({'Homogeneous 3×3': 'yes' if hh else 'no', 'Stable 2021–24': 'yes' if ee_ else 'no', 'n': int(m_.sum()),
                        **{n: accuracy_score(y[i24][m_], np.concatenate([P['test_T'], P['test_ST']])[m_]) for n, P in PRED.items()}})
    est = pd.DataFrame(est); est.to_csv(OUT_DIR / 'accuracy_by_stratum_2024.csv', index=False)
    print('Overall accuracy (OA) by stratum of the 2024 points'); display(est.round(3))

    ci, mc = [], []
    for cj in ['test_ST', 'test_ST_c']:
        yt = y[S[cj]]
        for leg, (cn, k, f) in LEGENDS.items():
            for n in names:
                (oa_l, oa_h), (f1_l, f1_h) = boot_ci(f(yt), f(PRED[n][cj]), k); mm = mets(f(yt), f(PRED[n][cj]), k)
                ci.append({'Set': cj, 'Legend': leg, 'Model': n, 'n': len(yt), 'OA': mm['OA'], 'OA_CI95': f'{oa_l:.3f}–{oa_h:.3f}',
                           'F1_macro': mm['F1_macro'], 'F1_CI95': f'{f1_l:.3f}–{f1_h:.3f}'})
            for m1, m2 in combinations(names, 2):
                b_, c_, p = mcnemar(f(yt), f(PRED[m1][cj]), f(PRED[m2][cj]))
                mc.append({'Set': cj, 'Legend': leg, 'A': m1, 'B': m2, 'b (only A correct)': b_, 'c (only B correct)': c_, 'p': p})
    ci = pd.DataFrame(ci); ci.to_csv(OUT_DIR / 'bootstrap_ci.csv', index=False)
    display(ci[ci.Legend == '8 classes'].round(3))
    mc = pd.DataFrame(mc); mc['p_Holm'] = mc.groupby(['Set', 'Legend'])['p'].transform(holm)
    mc['Conclusion (α=0.05)'] = np.where(mc.p_Holm < .05,
        np.where(mc['b (only A correct)'] > mc['c (only B correct)'], mc.A + ' > ' + mc.B, mc.B + ' > ' + mc.A),
        'no significant difference')
    mc.to_csv(OUT_DIR / 'mcnemar.csv', index=False)
    print('McNemar (Holm) · significant differences only:')
    display(mc[mc.p_Holm < .05][['Set', 'Legend', 'A', 'B', 'p_Holm', 'Conclusion (α=0.05)']]
            .style.format({'p_Holm': '{:.2e}'}))

    for cj in ['test_ST', 'test_ST_c']:
        f1c = pd.DataFrame({n: f1_score(y[S[cj]], PRED[n][cj], labels=range(K), average=None, zero_division=0)
                            for n in names}, index=CLASS_NAMES)
        f1c.to_csv(OUT_DIR / f'f1_by_class_{cj}.csv'); print(f'F1 by class · {cj}')
        display(f1c.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1, axis=None).format('{:.2f}'))
    plot_cm(PRED, y[S['test_ST']], CLASS_NAMES, K, 'test_ST', 'test_ST', 'confusion_matrices_test_ST.png')

    sub8 = res[(res.Legend == '8 classes') & res.Set.isin(['test_S', 'test_T', 'test_T_c', 'test_ST', 'test_ST_c'])]
    piv = sub8.pivot(index='Model', columns='Set', values='F1_macro').loc[names][['test_S', 'test_T_c', 'test_T', 'test_ST_c', 'test_ST']]
    ax = piv.plot.bar(rot=0, figsize=(10, 3.8), color=['#9ecae1', '#fdd0a2', '#fd8d3c', '#fcbba1', '#d62728'])
    ax.set_ylim(0, 1); ax.set_ylabel('Macro F1'); ax.grid(axis='y', alpha=.3); ax.legend(ncol=5, fontsize=8)
    ax.set_title('Generalization · _c = stable and homogeneous 2024 points only (comparable with training)')
    plt.tight_layout(); plt.savefig(OUT_DIR / 'model_comparison.png', dpi=200); plt.show()

    i24 = np.concatenate([S['test_T'], S['test_ST']])
    sz = pd.DataFrame({'szh': D['szh'][i24], 'y': y[i24],
                       **{n: np.concatenate([P['test_T'], P['test_ST']]) for n, P in PRED.items()}})
    by_sz = sz.groupby('szh').apply(lambda g: pd.Series({'n': len(g), **{n: (g[n] == g.y).mean() for n in names}}),
                                    include_groups=False)
    by_sz = by_sz[by_sz.n >= 30].sort_values('n', ascending=False)
    by_sz.to_csv(OUT_DIR / 'accuracy_by_subzone_2024.csv'); display(by_sz.round(3))

    best = res[(res.Legend == '8 classes') & (res.Set == 'test_ST')].sort_values('F1_macro', ascending=False).Model.iloc[0]
    STATE['best_model'] = best; save_state(); print('Best model (macro F1 test_ST, 8 classes):', best)
    mark_done('evaluate')

## 7 · 2024 / 2025 Maps and Interpretable Change

In [ ]:
import rasterio
from rasterio.transform import Affine
from numpy.lib.stride_tricks import sliding_window_view
from scipy import ndimage

def read_window(path):
    with rasterio.open(path) as src: arr, prof, desc = src.read(), src.profile, list(src.descriptions)
    f = arr[:C].astype('float32'); nod = f == NODATA_INT16; f /= BAND_SCALE[:, None, None]; f[nod] = 0
    extra = {d: arr[i] for i, d in enumerate(desc) if i >= C and d}
    return np.moveaxis(f, 0, -1), extra, ~nod.any(0), prof

def window_probs(name, feats, valid, stride, rows_per_chunk=8):
    r = PATCH // 2; Hh, Ww, _ = feats.shape
    pad = np.pad(feats, ((r, r), (r, r), (0, 0)), mode='reflect')
    rows = np.arange(Hh // stride) * stride + stride // 2
    cols = np.arange(Ww // stride) * stride + stride // 2
    P = np.full((len(rows), len(cols), K), np.nan, np.float32)
    for i0 in range(0, len(rows), rows_per_chunk):
        rr = rows[i0:i0 + rows_per_chunk]
        win = sliding_window_view(pad[rr[0]:rr[-1] + PATCH], (PATCH, PATCH), axis=(0, 1))[rr - rr[0]][:, cols]
        Xp = np.ascontiguousarray(np.moveaxis(win, 2, -1)).reshape(-1, PATCH, PATCH, C)
        p = probs_all(name, ((Xp - MEAN) / STD).astype('float32')).reshape(len(rr), len(cols), K)
        p[~valid[np.ix_(rr, cols)]] = np.nan; P[i0:i0 + len(rr)] = p
    return P

def to_cls(P):
    ok = ~np.isnan(P[..., 0])
    cls = np.where(ok, np.nan_to_num(P).argmax(-1), LABEL_NODATA).astype(np.uint8)
    conf = np.where(ok, np.nan_to_num(P).max(-1), 0).astype(np.float32)
    return cls, conf

def write_tif(path, arr, prof, stride, dtype='uint8', nodata=LABEL_NODATA):
    t = prof['transform']; p = {k: v for k, v in prof.items() if k not in ('blockxsize', 'blockysize', 'tiled')}
    p.update(count=1, dtype=dtype, nodata=nodata, compress='deflate', height=arr.shape[0], width=arr.shape[1],
             transform=Affine(t.a * stride, t.b, t.c, t.d, t.e * stride, t.f))
    with rasterio.open(path, 'w', **p) as dst: dst.write(arr.astype(dtype), 1)

def sub(a, s): c = s // 2; return a[c::s, c::s][: a.shape[0] // s, : a.shape[1] // s]

def mmu_filter(mask, min_px):
    lab, n = ndimage.label(mask, structure=np.ones((3, 3)))
    if n == 0: return mask
    size = ndimage.sum(mask, lab, index=np.arange(1, n + 1))
    keep = np.zeros(n + 1, bool); keep[1:] = size >= min_px
    return keep[lab]

def mb_rates(extra, s_):
    yrs = CFG['train_years'] + [CFG['test_year']]
    lay = [sub(extra['label'] if yy == CFG['test_year'] else extra[f'mb_{yy}'], s_) for yy in yrs]
    gross, pers = [], []
    for i in range(len(lay) - 1):
        ok = (lay[i] != LABEL_NODATA) & (lay[i + 1] != LABEL_NODATA)
        gross.append((lay[i][ok] != lay[i + 1][ok]).mean())
        if i + 2 < len(lay):
            ok2 = ok & (lay[i + 2] != LABEL_NODATA)
            pers.append(((lay[i] != lay[i + 1]) & (lay[i + 1] == lay[i + 2]))[ok2].mean())
    return float(np.mean(gross)), float(np.mean(pers)) if pers else np.nan

def em_priors(P, pi_tr, iters=200, tol=1e-6):
    pi = pi_tr.copy()
    for _ in range(iters):
        q = P * (pi / pi_tr); q /= q.sum(1, keepdims=True)
        new = q.mean(0)
        if np.abs(new - pi).max() < tol: return new
        pi = new
    return pi

def reweight(P, w):
    q = P * w; return q / q.sum(-1, keepdims=True)

def verdict(ratio):
    if not np.isfinite(ratio): return 'no reference'
    if ratio < 1 / 3: return 'underestimates'
    return 'consistent' if ratio <= 3 else ('doubtful' if ratio <= CFG['noise_ratio'] else 'noise')

if should_run('maps'):
    if 'MEAN' not in globals():
        nz = json.load(open(OUT_DIR / 'normalization.json')); MEAN, STD = np.array(nz['mean'], 'float32'), np.array(nz['std'], 'float32')
    if not CAL and CAL_FILE.exists(): CAL = json.loads(CAL_FILE.read_text())
    s_ = CFG['map_stride']; PIX_HA = (10 * s_) ** 2 / 1e4; MMU_PX = int(np.ceil(CFG['mmu_ha'] / PIX_HA))
    print(f'Output pixel: {10 * s_} m ({PIX_HA:.2f} ha) · minimum mapping unit: {CFG["mmu_ha"]} ha = {MMU_PX} pixels')
    NAMES = list(MODELS) + [k_ for k_, v in CAL.items() if isinstance(v, dict) and 'members' in v]
    PI_TR = (np.bincount(y[S['train']], minlength=K) / len(S['train'])) if 'S' in globals() else np.full(K, 1 / K)
    wins = sorted({Path(f).name.rsplit('_', 1)[0] for f in glob.glob(str(DATA_DIR / 'ventana_*_*.tif'))})
    MAPS, AGREE, MBR, PRI, AREA_CMP = {}, [], {}, [], []
    for win in wins:
        MAPS[win] = {}; PWY = {}
        for yr in [CFG['test_year'], CFG['pred_year']]:
            fl_ = sorted(glob.glob(str(DATA_DIR / f'{win}_{yr}*.tif')))
            if not fl_: continue
            feats, extra, valid, prof = read_window(fl_[0])
            no_s1 = np.all(feats[..., len(S2_FEATS):] == 0, axis=-1)
            if no_s1.mean() > 0.001: print(f'⚠ {win} {yr}: {100 * no_s1.mean():.1f} % without Sentinel-1 → unclassified')
            valid = valid & ~no_s1
            MAPS[win][f'rgb_{yr}'] = np.clip(sub(feats, s_)[..., [2, 1, 0]] / .15, 0, 1)
            if 'label' in extra:
                MAPS[win]['MapBiomas'] = sub(extra['label'], s_).astype(np.uint8)
                lastk = f"mb_{CFG['train_years'][-1]}"
                if lastk in extra: MAPS[win]['MapBiomas_prev'] = sub(extra[lastk], s_).astype(np.uint8)
                if all(f'mb_{yy}' in extra for yy in CFG['train_years']): MBR[win] = mb_rates(extra, s_)
            PW = {}
            for n in MODELS:
                t0 = time.time(); PW[n] = window_probs(n, feats, valid, s_)
                print(f'{win} · {n} · {yr}: {PW[n].shape[:2]} in {time.time() - t0:.0f} s')
            nanmask = np.isnan(PW[next(iter(MODELS))][..., 0])
            for en in NAMES:
                if en in MODELS: continue
                PW[en] = calibrate(np.nan_to_num(np.mean([PW[m] for m in CAL[en]['members']], axis=0), nan=1 / K), CAL[en]['T'])
                PW[en][nanmask] = np.nan
            PWY[yr] = PW
        ref = MAPS[win].get('MapBiomas')
        for n in NAMES:
            allp = np.concatenate([PWY[yr][n].reshape(-1, K) for yr in PWY]); allp = allp[~np.isnan(allp[:, 0])]
            if len(allp) > 300_000: allp = allp[np.random.default_rng(0).choice(len(allp), 300_000, replace=False)]
            mbp = MAPS[win].get('MapBiomas_prev')
            if CFG['prior_corr'] == 'em':
                pi_est = em_priors(allp, PI_TR)
            elif CFG['prior_corr'] == 'mb_prev' and mbp is not None:
                pi_est = np.bincount(mbp[mbp != LABEL_NODATA], minlength=K)[:K] / max(1, (mbp != LABEL_NODATA).sum())
            else:
                pi_est = PI_TR
            pi_est = np.maximum(pi_est, CFG['em_floor']); pi_est /= pi_est.sum()
            w = np.clip((pi_est / PI_TR) ** CFG['prior_alpha'], 1 / CFG['em_wmax'], CFG['em_wmax'])
            PRI.append({'Window': win, 'Model': n, **{cn: 100 * pi_est[k] for k, cn in enumerate(CLASS_NAMES)}})
            for yr in PWY:
                cls_raw, _ = to_cls(PWY[yr][n]); cls_c, conf_c = to_cls(reweight(PWY[yr][n], w))
                MAPS[win][f'{n}_raw_{yr}'] = cls_raw
                MAPS[win][f'{n}_{yr}'], MAPS[win][f'{n}_{yr}_conf'] = cls_c, conf_c
                write_tif(MAP_DIR / f'{win}_{n}_{yr}_s{s_}.tif', cls_c, prof, s_)
                write_tif(MAP_DIR / f'{win}_{n}_{yr}_s{s_}_conf.tif', np.round(conf_c * 100), prof, s_, nodata=0)
                if yr == CFG['test_year'] and ref is not None:
                    for tag, cl_ in [('raw', cls_raw), ('corrected', cls_c)]:
                        ok = (ref != LABEL_NODATA) & (cl_ != LABEL_NODATA)
                        AGREE.append({'Window': win, 'Model': n, 'Map': tag,
                                      **{f'{k}_8': v for k, v in mets(ref[ok], cl_[ok]).items()},
                                      **{f'{k}_6': v for k, v in mets(TO6[ref[ok]], TO6[cl_[ok]], K6).items()}})
                    ok = (ref != LABEL_NODATA) & (cls_c != LABEL_NODATA)
                    for k, cn in enumerate(CLASS_NAMES):
                        mb_ha = (ref[ok] == k).sum() * PIX_HA
                        AREA_CMP.append({'Window': win, 'Model': n, 'Class': cn, 'ha_MapBiomas': mb_ha,
                                         'ha_raw': (cls_raw[ok] == k).sum() * PIX_HA, 'ha_corrected': (cls_c[ok] == k).sum() * PIX_HA})
    agree = pd.DataFrame(AGREE); agree.to_csv(OUT_DIR / 'mapbiomas_agreement_2024_windows.csv', index=False)
    if len(agree): display(agree.pivot_table(index=['Model'], columns=['Map', 'Window'], values='OA_8').round(3))
    pd.DataFrame(PRI).to_csv(OUT_DIR / 'window_priors.csv', index=False)
    ac = pd.DataFrame(AREA_CMP)
    if len(ac):
        tot = ac.groupby(['Window', 'Model']).ha_MapBiomas.transform('sum')
        ac['bias_raw_pp'] = 100 * (ac.ha_raw - ac.ha_MapBiomas) / tot
        ac['bias_corrected_pp'] = 100 * (ac.ha_corrected - ac.ha_MapBiomas) / tot
        ac.to_csv(OUT_DIR / 'areas_vs_mapbiomas_2024.csv', index=False)
        summary = ac.groupby(['Model']).agg(abs_bias_raw=('bias_raw_pp', lambda v: v.abs().mean()),
                                            abs_bias_corrected=('bias_corrected_pp', lambda v: v.abs().mean()))
        print('Mean absolute area bias per class vs MapBiomas 2024 (percentage points of the window area)')
        display(summary.round(2))
        print('Per-class detail (pp) · positive = the model overestimates the class')
        display(ac.pivot_table(index='Class', columns='Model', values='bias_corrected_pp', aggfunc='mean').round(1)
                  .style.background_gradient(cmap='RdBu_r', vmin=-10, vmax=10, axis=None).format('{:+.1f}'))

    ty_, py_, ly_ = CFG['test_year'], CFG['pred_year'], CFG['train_years'][-1]
    for win, M in MAPS.items():
        cols = ['rgb', 'mb'] + NAMES
        fig, axs = plt.subplots(2, len(cols), figsize=(2.8 * len(cols), 6.2))
        for r_, yr in enumerate([ty_, py_]):
            for c_, key in enumerate(cols):
                ax = axs[r_, c_]; ax.axis('off')
                if key == 'rgb':
                    a, t, is_rgb = M.get(f'rgb_{yr}'), f'RGB {yr}', True
                elif key == 'mb':
                    a, t, is_rgb = (M.get('MapBiomas'), f'MapBiomas {ty_}', False) if yr == ty_ else \
                                   (M.get('MapBiomas_prev'), f'MapBiomas {ly_} (ref.)', False)
                else:
                    a, t, is_rgb = M.get(f'{key}_{yr}'), f'{key} {yr}', False
                if a is None: continue
                ax.imshow(a) if is_rgb else ax.imshow(np.ma.masked_equal(a, LABEL_NODATA), cmap=CMAP, norm=NORM, interpolation='nearest')
                ax.set_title(t, fontsize=8)
        fig.suptitle(win.replace('ventana_', '').replace('_', ' ') + f" · prior: {CFG['prior_corr'] or 'none'} (α = {CFG['prior_alpha']})", y=1.0)
        fig.legend(handles=LEGEND, loc='lower center', ncol=8, fontsize=7, bbox_to_anchor=(.5, -.03))
        plt.tight_layout(); plt.savefig(OUT_DIR / f'maps_{win}.png', dpi=200, bbox_inches='tight'); plt.show()

    CH, AREAS = [], []
    BEST = STATE.get('best_model', NAMES[0])
    for win, M in MAPS.items():
        gross, pers = MBR.get(win, (np.nan, np.nan))
        for n in NAMES:
            a, b = M.get(f"{n}_{CFG['test_year']}"), M.get(f"{n}_{CFG['pred_year']}")
            if a is None or b is None: continue
            th = CAL.get(n, {}).get('threshold', CFG['conf_min'])
            ca, cb = M[f"{n}_{CFG['test_year']}_conf"], M[f"{n}_{CFG['pred_year']}_conf"]
            ok = (a != LABEL_NODATA) & (b != LABEL_NODATA)
            raw = ok & (a != b); hi = ok & (ca >= th) & (cb >= th)
            filt = mmu_filter(raw & hi, MMU_PX)
            r_raw, r_f = raw.sum() / ok.sum(), filt.sum() / ok.sum()
            row = {'Window': win, 'Model': n, 'Threshold': th, 'High_conf_area_%': 100 * hi.sum() / ok.sum(),
                   'MB_gross_%/yr': 100 * gross, 'MB_persistent_%/yr': 100 * pers,
                   'Model_raw_%': 100 * r_raw, 'Model_filtered_%': 100 * r_f,
                   'Ratio_filtered/persistent': r_f / pers if pers else np.nan}
            row['Verdict'] = verdict(row['Ratio_filtered/persistent']); CH.append(row)
            write_tif(MAP_DIR / f'{win}_{n}_filtered_change_s{s_}.tif', np.where(ok, filt, 255), prof, s_)
            b_f = np.where(filt, b, a)
            for k, cn in enumerate(CLASS_NAMES):
                AREAS.append({'Window': win, 'Model': n, 'Class': cn, 'ha_2024': (a[ok] == k).sum() * PIX_HA,
                              'ha_2025_raw': (b[ok] == k).sum() * PIX_HA, 'ha_2025_filtered': (b_f[ok] == k).sum() * PIX_HA})
            if n == BEST:
                tr = pd.crosstab(pd.Categorical(a[filt], range(K)), pd.Categorical(b[filt], range(K)), dropna=False) * PIX_HA
                tr.index = tr.columns = CLASS_NAMES; tr.round(1).to_csv(OUT_DIR / f'filtered_transitions_{win}_{BEST}.csv')
                fig, axs = plt.subplots(1, 4, figsize=(13, 3.8))
                for ax, (t, im) in zip(axs, [(f'{BEST} 2024', a), (f'{BEST} 2025', b)]):
                    ax.imshow(np.ma.masked_equal(im, LABEL_NODATA), cmap=CMAP, norm=NORM, interpolation='nearest'); ax.set_title(t)
                axs[2].imshow(np.where(ok, raw.astype(float), np.nan), cmap='Reds', vmin=0, vmax=1); axs[2].set_title(f'Raw ({100 * r_raw:.1f} %)')
                axs[3].imshow(np.where(ok, filt.astype(float), np.nan), cmap='Reds', vmin=0, vmax=1)
                axs[3].set_title(f'Filtered {100 * r_f:.2f} % · MB persist. {100 * pers:.2f} %', fontsize=9)
                for ax in axs: ax.axis('off')
                plt.suptitle(win); plt.tight_layout(); plt.savefig(OUT_DIR / f'change_{win}_{BEST}.png', dpi=200, bbox_inches='tight'); plt.show()
    ch = pd.DataFrame(CH); ch.to_csv(OUT_DIR / 'change_vs_mapbiomas.csv', index=False); display(ch.round(2))
    areas = pd.DataFrame(AREAS); areas['filtered_change_ha'] = areas.ha_2025_filtered - areas.ha_2024
    areas.to_csv(OUT_DIR / 'areas_2024_2025.csv', index=False)
    print('Flooded grassland 2024 (ha, corrected maps) — spread across models = area uncertainty')
    hi_ = areas[areas.Class == 'Flooded grassland'].pivot(index='Model', columns='Window', values='ha_2024')
    if len(ac): hi_.loc['MapBiomas'] = ac[(ac.Class == 'Flooded grassland')].groupby('Window').ha_MapBiomas.first()
    display(hi_.round(0))
    mark_done('maps')

### 7b · Class Areas: Prior-Corrected Maps vs MapBiomas

In [ ]:
if should_run('areas'):
    ty_, py_ = CFG['test_year'], CFG['pred_year']; rows = []
    for win, M in MAPS.items():
        ref = M.get('MapBiomas')
        for n in NAMES:
            a_raw, a_c, b_c = M.get(f'{n}_raw_{ty_}'), M.get(f'{n}_{ty_}'), M.get(f'{n}_{py_}')
            if a_c is None or ref is None: continue
            ok = (ref != LABEL_NODATA) & (a_c != LABEL_NODATA)
            for k, cn in enumerate(CLASS_NAMES):
                rows.append({'Window': win, 'Model': n, 'Class': cn,
                             'ha_MapBiomas_2024': (ref[ok] == k).sum() * PIX_HA,
                             'ha_raw_2024': (a_raw[ok] == k).sum() * PIX_HA,
                             'ha_prior_2024': (a_c[ok] == k).sum() * PIX_HA})
    TA = pd.DataFrame(rows)
    if 'areas' in globals() and len(areas):
        TA = TA.merge(areas[['Window', 'Model', 'Class', 'ha_2025_filtered']].rename(columns={'ha_2025_filtered': 'ha_prior_2025'}),
                      on=['Window', 'Model', 'Class'], how='left')
        TA['change_2024_2025_ha'] = TA.ha_prior_2025 - TA.ha_prior_2024
    tot = TA.groupby(['Window', 'Model']).ha_MapBiomas_2024.transform('sum')
    TA['bias_raw_pp'] = 100 * (TA.ha_raw_2024 - TA.ha_MapBiomas_2024) / tot
    TA['bias_prior_pp'] = 100 * (TA.ha_prior_2024 - TA.ha_MapBiomas_2024) / tot
    TA.to_csv(OUT_DIR / 'areas_prior_vs_mapbiomas.csv', index=False)

    print('Mean absolute area bias per class vs MapBiomas 2024 (pp of the window area)')
    display(TA.groupby('Model')[['bias_raw_pp', 'bias_prior_pp']].apply(lambda d: d.abs().mean())
              .rename(columns={'bias_raw_pp': 'raw map', 'bias_prior_pp': 'prior-corrected map'}).round(2))
    print('Per-class bias of the prior-corrected map (pp) · mean over windows')
    display(TA.pivot_table(index='Class', columns='Model', values='bias_prior_pp', aggfunc='mean').reindex(CLASS_NAMES).round(1)
              .style.background_gradient(cmap='RdBu_r', vmin=-8, vmax=8, axis=None).format('{:+.1f}'))

    FINAL = CFG['final_model'] if CFG['final_model'] != 'auto' else STATE.get('best_model', NAMES[0])
    cols_ = [c_ for c_ in ['ha_MapBiomas_2024', 'ha_raw_2024', 'ha_prior_2024', 'ha_prior_2025', 'change_2024_2025_ha', 'bias_prior_pp'] if c_ in TA]
    for win in MAPS:
        d = TA[(TA.Model == FINAL) & (TA.Window == win)].set_index('Class').reindex(CLASS_NAMES)
        if d.empty: continue
        print(f"Areas (ha) · {win.replace('ventana_', '')} · {FINAL}"); display(d[cols_].round(1))
        fig, ax = plt.subplots(figsize=(11, 3.8)); xk = np.arange(K); wb = .2
        ax.bar(xk - 1.5 * wb, d.ha_MapBiomas_2024, wb, color='#636363', label=f'MapBiomas {ty_}')
        ax.bar(xk - .5 * wb, d.ha_raw_2024, wb, color='#bdbdbd', label=f'{FINAL} raw {ty_}')
        ax.bar(xk + .5 * wb, d.ha_prior_2024, wb, color='#3182bd', label=f'{FINAL} with prior {ty_}')
        if 'ha_prior_2025' in d: ax.bar(xk + 1.5 * wb, d.ha_prior_2025, wb, color='#e6550d', label=f'{FINAL} with prior {py_} (filtered change)')
        ax.set_xticks(xk); ax.set_xticklabels(CLASS_NAMES, rotation=20, fontsize=8); ax.set_ylabel('ha')
        ax.set_title(f"{win.replace('ventana_', '').replace('_', ' ')} · areas by class"); ax.legend(fontsize=7, ncol=4)
        plt.tight_layout(); plt.savefig(OUT_DIR / f'areas_{win}.png', dpi=200); plt.show()
    mark_done('areas')

### 7c · Final 10 m Map

In [ ]:
if should_run('final_map'):
    if 'MEAN' not in globals():
        nz = json.load(open(OUT_DIR / 'normalization.json')); MEAN, STD = np.array(nz['mean'], 'float32'), np.array(nz['std'], 'float32')
    FINAL = CFG['final_model'] if CFG['final_model'] != 'auto' else STATE.get('best_model', next(iter(MODELS)))
    fs = CFG['final_stride']; members = CAL[FINAL]['members'] if FINAL in CAL and 'members' in CAL[FINAL] else [FINAL]
    print(f'Final map: {FINAL} ({members}) · pixel {10 * fs} m')
    wins = sorted({Path(f).name.rsplit('_', 1)[0] for f in glob.glob(str(DATA_DIR / 'ventana_*_*.tif'))})
    for win in wins:
        FIG = {}; w = np.ones(K)
        for yr in [CFG['test_year'], CFG['pred_year']]:
            fl_ = sorted(glob.glob(str(DATA_DIR / f'{win}_{yr}*.tif')))
            if not fl_: continue
            feats, extra, valid, prof = read_window(fl_[0])
            valid = valid & ~np.all(feats[..., len(S2_FEATS):] == 0, axis=-1)
            if yr == CFG['test_year']:
                FIG['mb'] = sub(extra['label'], fs) if 'label' in extra else None
                lk = f"mb_{CFG['train_years'][-1]}"
                if CFG['prior_corr'] == 'mb_prev' and lk in extra:
                    mbp = sub(extra[lk], fs); pi = np.bincount(mbp[mbp != LABEL_NODATA], minlength=K)[:K] / max(1, (mbp != LABEL_NODATA).sum())
                    pi = np.maximum(pi, CFG['em_floor']); pi /= pi.sum()
                    w = np.clip((pi / PI_TR) ** CFG['prior_alpha'], 1 / CFG['em_wmax'], CFG['em_wmax'])
            t0 = time.time()
            P = np.mean([window_probs(m, feats, valid, fs) for m in members], axis=0)
            if FINAL in CAL and 'members' in CAL[FINAL]:
                nm = np.isnan(P[..., 0]); P = calibrate(np.nan_to_num(P, nan=1 / K), CAL[FINAL]['T']); P[nm] = np.nan
            cls, conf = to_cls(reweight(P, w))
            write_tif(MAP_DIR / f'FINAL_{win}_{FINAL}_{yr}_{10 * fs}m.tif', cls, prof, fs)
            write_tif(MAP_DIR / f'FINAL_{win}_{FINAL}_{yr}_{10 * fs}m_conf.tif', np.round(conf * 100), prof, fs, nodata=0)
            FIG[yr] = cls; FIG[f'rgb_{yr}'] = np.clip(sub(feats, fs)[..., [2, 1, 0]] / .15, 0, 1)
            print(f'  {win} · {yr}: {cls.shape} in {(time.time() - t0) / 60:.1f} min')
        fig, axs = plt.subplots(1, 5, figsize=(17, 3.8))
        pan = [(FIG.get(f"rgb_{CFG['test_year']}"), 'RGB 2024', 1), (FIG.get('mb'), 'MapBiomas 2024', 0),
               (FIG.get(CFG['test_year']), f'{FINAL} 2024', 0), (FIG.get(CFG['pred_year']), f'{FINAL} 2025', 0),
               (FIG.get(f"rgb_{CFG['pred_year']}"), 'RGB 2025', 1)]
        for ax, (a, t, is_rgb) in zip(axs, pan):
            ax.axis('off')
            if a is None: continue
            ax.imshow(a) if is_rgb else ax.imshow(np.ma.masked_equal(a, LABEL_NODATA), cmap=CMAP, norm=NORM, interpolation='nearest')
            ax.set_title(t, fontsize=9)
        fig.suptitle(f"{win.replace('ventana_', '').replace('_', ' ')} · {10 * fs} m"); fig.legend(handles=LEGEND, loc='lower center', ncol=8, fontsize=7, bbox_to_anchor=(.5, -.06))
        plt.savefig(OUT_DIR / f'FINAL_{win}_{FINAL}_{10 * fs}m.png', dpi=250, bbox_inches='tight'); plt.show()
    mark_done('final_map')

## 8 · Workflow Summary

In [ ]:
print('Completed steps:')
for k, v in STATE['steps'].items(): print(f'  {k:12s} {v}')
if STATE.get('best_model'): print('Best model:', STATE['best_model'])
print('\nOutputs in', OUT_DIR)
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file(): print('  ', p.relative_to(OUT_DIR))
if IN_COLAB and 'train' in STATE['steps']:
    import shutil
    zipf = shutil.make_archive('/content/results_colab', 'zip', OUT_DIR)
    print('\nDownloading', zipf, '→ unzip it into the results_v6/ folder on the PC and use PROFILE = "pc_maps"')
    files.download(zipf)